In [ ]:
# === Setup ===
# Runtime: <1 minute fast, <2 minutes full on a typical CPU (estimate).
# Hardware: CPU ok; no GPU required.
# Network: none; all datasets are generated locally.
# Competition-safe: general profile; check the actual contest package/data policy.
# Cẩm nang P08: NumPy, pandas, sklearn, Matplotlib, joblib; no package installation.
import os
import random
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
random.seed(42)
np.random.seed(42)
FAST = os.environ.get('OAI_FAST_MODE', '0') == '1'
rng = np.random.default_rng(42)
OUT = Path('outputs')
OUT.mkdir(exist_ok=True)


# Validation — Đo đúng khả năng tổng quát hóa

Starter: baseline chạy hết; hoàn thành TODO trước khi đọc solution.

## Data → EDA

Label được gán ngẫu nhiên cho mỗi người, các lần đo gần giống fingerprint người đó. Cơ chế tạo dữ liệu chủ đích loại tín hiệu tổng quát; mô hình chỉ có thể nhớ người đã gặp.

In [ ]:
from sklearn.model_selection import (GroupKFold, GroupShuffleSplit,
    StratifiedKFold, KFold, TimeSeriesSplit)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score

group_count = 120 if FAST else 240
groups = np.repeat(np.arange(group_count), 4)
fingerprints = rng.normal(size=(group_count, 10))
group_labels = rng.integers(0, 2, size=group_count)
X = fingerprints[groups] + rng.normal(0, 0.01, (len(groups), 10))
y = group_labels[groups]
test_groups = np.repeat(np.arange(32), 4)
X_test = rng.normal(size=(32, 10))[test_groups] + rng.normal(0, 0.01, (128, 10))
test_ids = np.array([f'test_{i}' for i in range(len(X_test))])

def build_model():
    """Return an unfitted nearest-neighbor pipeline for X (n, 10)."""
    return make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=1))

print('labeled shape:', X.shape)
print('class counts:', np.bincount(y))
assert np.isfinite(X).all()


## Preprocess → Model → Train

Scaler phải fit lại trong mỗi fold. Dự đoán row split và group split sẽ cho kết luận khác nhau.

In [ ]:
tr, va = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
              .split(X, y, groups))
assert set(groups[tr]).isdisjoint(groups[va])
model = build_model().fit(X[tr], y[tr])
# TODO: replace this single hold-out report with GroupKFold and exactly-once OOF.
# TODO: compare row CV under the same estimator without selecting the optimistic split.


## Evaluate

In [ ]:
score = f1_score(y[va], model.predict(X[va]), labels=[0, 1], average='macro', zero_division=0)
print('group hold-out macro_f1:', round(score, 6))
assert 0 <= score <= 1


## Submit

Giữ mục tiêu người mới. Refit baseline trên labeled data sau khi đánh giá; public test không có label.

In [ ]:
model = build_model().fit(X, y)
test_pred = model.predict(X_test)


In [ ]:
# WHY: validate the file read from disk, not only the in-memory frame.
submission = pd.DataFrame({'id': test_ids, 'label': test_pred})
assert submission.columns.tolist() == ['id', 'label']
assert len(submission) == len(test_ids)
assert submission['id'].is_unique
assert submission['label'].isin([0, 1]).all()
submission.to_csv(OUT / 'submission_starter.csv', index=False)
reloaded = pd.read_csv(OUT / 'submission_starter.csv')
assert reloaded['id'].tolist() == list(test_ids)
assert reloaded['label'].tolist() == list(test_pred)
print('submission rows:', len(reloaded))


## Postmortem

Ghi lại đơn vị dự đoán, group overlap, coverage và giới hạn generalization. Không gọi điểm CV thấp là lỗi code khi dữ liệu không có tín hiệu cho người mới.